# 08주차: PyTorch 이미지 분류 최적화

## 학습 목표
7주차에서 사용한 `ImprovedCNN`과 같은 블록 구조를 이 노트북 안에서 독립적으로 다시 정의하고, 학습률, 옵티마이저(SGD·Adam), 배치 정규화, 드롭아웃, Early Stopping이 CIFAR-10 분류 성능에 어떤 영향을 주는지 순서대로 관찰합니다. 여러 설정을 짧게 비교하는 실험과, 찾은 설정을 실제 규모로 학습하는 최종 실험을 구분해서 진행합니다.

Colab에서는 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택하세요. GPU가 없어도 CPU에서 실행되지만 학습이 오래 걸릴 수 있습니다. CIFAR-10은 torchvision이 자동으로 내려받으므로 Google Drive 연결, Drive 마운트, 파일 업로드는 필요하지 않습니다. 이 노트북은 6·7주차 파일을 불러오지 않고 데이터 다운로드, 분할, 모델, 학습 함수를 모두 새로 정의합니다.

## 관찰 질문
- 학습률이 너무 작거나 너무 크면 짧은 학습 안에서 어떤 신호로 드러날까요?
- 배치 정규화와 드롭아웃은 왜 같이 켜졌을 때 학습 안정성과 과적합 방지에 함께 도움이 될까요?
- Early Stopping은 왜 "가장 낮은 검증 손실"을 기준으로 멈추고, 마지막 에포크의 가중치를 그대로 쓰지 않을까요?


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 장치:", device)

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)
base_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
augment_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

full_train_augment = datasets.CIFAR10(root="./data", train=True, download=True, transform=augment_transform)
full_train_base = datasets.CIFAR10(root="./data", train=True, download=True, transform=base_transform)
test_dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=base_transform)
class_names = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

split_generator = torch.Generator().manual_seed(42)
permutation = torch.randperm(len(full_train_base), generator=split_generator).tolist()
train_indices = permutation[:45000]
val_indices = permutation[45000:]
print("고정 분할 크기(훈련/검증):", len(train_indices), len(val_indices))

train_dataset = Subset(full_train_augment, train_indices)
val_dataset = Subset(full_train_base, val_indices)

batch_size = 128
loader_options = {"batch_size": batch_size, "num_workers": 2, "pin_memory": torch.cuda.is_available()}
train_loader = DataLoader(train_dataset, shuffle=True, **loader_options)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_options)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_options)
print("전체 데이터 수(훈련/검증/테스트):", len(train_dataset), len(val_dataset), len(test_dataset))


## ImprovedCNN 구조 재정의
7주차에서 사용한 것과 같은 블록 구조(합성곱 두 번 반복 후 풀링, 채널 32→64→128)를 이 노트북 안에서 다시 정의합니다. `use_batchnorm`을 켜면 각 합성곱 뒤에 `nn.BatchNorm2d`가 추가되고, `dropout`은 분류기 앞단의 `nn.Dropout` 비율을 결정합니다. 이번 주차의 모든 실험은 이 하나의 구조에서 옵티마이저·학습률·정규화 설정만 바꾸며 진행합니다.


In [ ]:
class ImprovedCNN(nn.Module):
    def __init__(self, use_batchnorm=False, dropout=0.0):
        super().__init__()
        def block(in_ch, out_ch):
            layers = [nn.Conv2d(in_ch, out_ch, 3, padding=1)]
            if use_batchnorm:
                layers.append(nn.BatchNorm2d(out_ch))
            layers += [nn.ReLU(), nn.Conv2d(out_ch, out_ch, 3, padding=1)]
            if use_batchnorm:
                layers.append(nn.BatchNorm2d(out_ch))
            layers += [nn.ReLU(), nn.MaxPool2d(2)]
            return layers
        self.features = nn.Sequential(
            *block(3, 32), *block(32, 64), *block(64, 128),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

sample_images, sample_labels = next(iter(train_loader))
with torch.no_grad():
    shape_logits = ImprovedCNN(use_batchnorm=True, dropout=0.3).to(device)(sample_images.to(device))
print("입력 텐서 shape:", sample_images.shape)
print("출력 텐서 shape:", shape_logits.shape)


## 공통 학습·평가 함수
`train_one_epoch`, `evaluate`, `fit`은 모델·데이터로더·옵티마이저·`device`를 인자로 받는 형태로 정의합니다. 이번 주차의 모든 비교 실험이 이 함수들을 그대로 재사용하므로, 학습 절차 자체는 고정한 채 하이퍼파라미터와 데이터 조건만 바꾸어 관찰할 수 있습니다.


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for batch_images, batch_labels in loader:
        batch_images, batch_labels = batch_images.to(device), batch_labels.to(device)
        optimizer.zero_grad()
        logits = model(batch_images)
        loss = criterion(logits, batch_labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * batch_labels.size(0)
        correct += (logits.argmax(dim=1) == batch_labels).sum().item()
        total += batch_labels.size(0)
    return loss_sum / total, 100 * correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for batch_images, batch_labels in loader:
            batch_images, batch_labels = batch_images.to(device), batch_labels.to(device)
            logits = model(batch_images)
            loss_sum += criterion(logits, batch_labels).item() * batch_labels.size(0)
            correct += (logits.argmax(dim=1) == batch_labels).sum().item()
            total += batch_labels.size(0)
    return loss_sum / total, 100 * correct / total

def fit(model, train_loader, val_loader, criterion, optimizer, device, epochs):
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    for epoch in range(1, epochs + 1):
        train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_accuracy = evaluate(model, val_loader, criterion, device)
        for key, value in [("train_loss", train_loss), ("train_acc", train_accuracy), ("val_loss", val_loss), ("val_acc", val_accuracy)]:
            history[key].append(value)
        print(f"Epoch {epoch}/{epochs}: train={train_accuracy:.2f}%, val={val_accuracy:.2f}%")
    return history

def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["train_loss"], label="train")
    axes[0].plot(history["val_loss"], label="validation")
    axes[0].set_title(f"{title} Loss")
    axes[0].legend()
    axes[1].plot(history["train_acc"], label="train")
    axes[1].plot(history["val_acc"], label="validation")
    axes[1].set_title(f"{title} Accuracy")
    axes[1].legend()
    plt.show()

def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

criterion = nn.CrossEntropyLoss()


## 짧은 비교 실험을 위한 고정 부분집합
학습률과 옵티마이저를 비교할 때마다 45,000개 전체 훈련 데이터로 학습하면 실습 시간 안에 여러 설정을 비교하기 어렵습니다. 이 노트북은 실행마다 데이터 양을 몰래 바꾸는 숨겨진 전역 개발용 단축 스위치를 두지 않고, 처음부터 계획된 **고정 훈련 인덱스 12,000개와 고정 검증 인덱스 3,000개**만 잘라 짧은 비교 전용 데이터셋(`short_train_loader`, `short_val_loader`)을 만듭니다. 이 인덱스는 `train_indices`, `val_indices`의 앞부분을 그대로 잘라낸 것이므로 실행할 때마다 항상 같은 이미지가 선택되며, 학습률 비교와 옵티마이저 비교 모두 이 동일한 데이터셋을 사용합니다. 이후 최종 모델 학습(BatchNorm·Dropout·Early Stopping 적용)에서는 이 부분집합이 아니라 전체 훈련 데이터(`train_loader`, 45,000개)를 사용합니다.

### 관찰 질문
- 왜 "여러 설정을 짧게 반복 비교"하는 실험에는 작은 고정 부분집합이 적합하고, 최종 모델 학습에는 전체 데이터가 필요할까요?
- 부분집합 크기(12,000/3,000)가 고정되어 있지 않고 매번 랜덤하게 바뀐다면 비교 결과를 신뢰하기 어려운 이유는 무엇일까요?


In [ ]:
short_train_indices = train_indices[:12000]
short_val_indices = val_indices[:3000]
print("짧은 비교 실험용 고정 인덱스 수(훈련/검증):", len(short_train_indices), len(short_val_indices))

short_train_dataset = Subset(full_train_augment, short_train_indices)
short_val_dataset = Subset(full_train_base, short_val_indices)
short_train_loader = DataLoader(short_train_dataset, shuffle=True, **loader_options)
short_val_loader = DataLoader(short_val_dataset, shuffle=False, **loader_options)
print("짧은 비교 데이터 수(훈련/검증):", len(short_train_dataset), len(short_val_dataset))


## 학습률 비교
같은 초기 시드(`seed_everything(42)`)와 같은 모델 구조(`ImprovedCNN(use_batchnorm=False, dropout=0.0)`)에서 학습률만 `1e-4`, `1e-3`, `1e-2`로 바꾸어 각각 3에포크 학습합니다. 짧은 비교용 고정 데이터셋(`short_train_loader`, `short_val_loader`)을 사용하므로 조건 차이는 오직 학습률뿐입니다.

### 관찰 질문
- 학습률이 너무 작으면(`1e-4`) 손실과 검증 정확도는 3에포크 안에 얼마나 움직일까요?
- 학습률이 너무 크면(`1e-2`) 손실 곡선이 들쭉날쭉하거나 발산하는 신호가 보이나요?


In [ ]:
learning_rates = [1e-4, 1e-3, 1e-2]
lr_histories = {}
for lr in learning_rates:
    seed_everything(42)
    model = ImprovedCNN(use_batchnorm=False, dropout=0.0).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    lr_histories[lr] = fit(
        model, short_train_loader, short_val_loader,
        criterion, optimizer, device, epochs=3
    )


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for lr, history in lr_histories.items():
    axes[0].plot(history["val_loss"], label=f"lr={lr}")
    axes[1].plot(history["val_acc"], label=f"lr={lr}")
axes[0].set_title("Validation Loss by Learning Rate")
axes[0].legend()
axes[1].set_title("Validation Accuracy by Learning Rate")
axes[1].legend()
plt.show()

for lr, history in lr_histories.items():
    print(f"lr={lr}: 최종 검증 정확도={history['val_acc'][-1]:.2f}%, 최종 검증 손실={history['val_loss'][-1]:.4f}")


### 학생 활동
1. 위 결과에서 어떤 학습률이 3에포크 안에 가장 빠르게 검증 정확도를 올렸는지 확인하세요.
2. `1e-2`처럼 큰 학습률에서 손실이 들쭉날쭉하거나 오히려 커지는 구간이 있다면, 왜 그런 현상이 나타나는지 적어보세요.
3. 이 결과가 "1e-3이 항상 최선"이라는 뜻은 아닙니다. 데이터셋, 모델 구조, 배치 크기가 달라지면 최적 학습률도 달라질 수 있음을 유의하세요.


## SGD와 Adam 비교
같은 초기 시드, 같은 모델 구조, 같은 짧은 비교용 데이터셋에서 `SGD(lr=0.03, momentum=0.9)`와 `Adam(lr=1e-3)`을 각각 3에포크 학습해 비교합니다.

### 관찰 질문
- 두 옵티마이저 중 3에포크 안에서 검증 정확도가 더 빨리 오르는 쪽은 어디인가요?
- 이 비교는 짧은 3에포크 실행 한 번의 결과입니다. "SGD가 항상 느리다"거나 "Adam이 항상 낫다"처럼 일반화해도 될까요?


In [ ]:
optimizer_configs = {
    "SGD": lambda params: torch.optim.SGD(params, lr=0.03, momentum=0.9),
    "Adam": lambda params: torch.optim.Adam(params, lr=1e-3),
}
optimizer_histories = {}
for name, make_optimizer in optimizer_configs.items():
    seed_everything(42)
    model = ImprovedCNN(use_batchnorm=False, dropout=0.0).to(device)
    optimizer = make_optimizer(model.parameters())
    optimizer_histories[name] = fit(
        model, short_train_loader, short_val_loader,
        criterion, optimizer, device, epochs=3
    )

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, history in optimizer_histories.items():
    axes[0].plot(history["val_loss"], label=name)
    axes[1].plot(history["val_acc"], label=name)
axes[0].set_title("Validation Loss by Optimizer")
axes[0].legend()
axes[1].set_title("Validation Accuracy by Optimizer")
axes[1].legend()
plt.show()

for name, history in optimizer_histories.items():
    print(f"{name}: 최종 검증 정확도={history['val_acc'][-1]:.2f}%, 최종 검증 손실={history['val_loss'][-1]:.4f}")


## BatchNorm·Dropout 켜기 전/후 비교
학습률·옵티마이저 비교와 같은 짧은 비교용 고정 데이터셋(`short_train_loader`, `short_val_loader`)에서, 같은 초기 시드로 `ImprovedCNN(use_batchnorm=False, dropout=0.0)`(끄기)와 `ImprovedCNN(use_batchnorm=True, dropout=0.3)`(켜기)를 각각 3에포크 학습해 검증 손실·정확도를 비교합니다. 옵티마이저는 두 경우 모두 `Adam(lr=1e-3)`으로 고정합니다.

### 관찰 질문
- 배치 정규화와 드롭아웃을 함께 켰을 때 3에포크라는 짧은 구간 안에서도 검증 손실·정확도가 다르게 움직이나요?
- 이 비교는 3에포크만 본 결과입니다. 에포크를 늘리면 두 설정의 차이가 더 커질지 작아질지 예상해 보세요.


In [ ]:
bn_dropout_configs = {
    "BN off / Dropout 0.0": lambda: ImprovedCNN(use_batchnorm=False, dropout=0.0),
    "BN on / Dropout 0.3": lambda: ImprovedCNN(use_batchnorm=True, dropout=0.3),
}
bn_dropout_histories = {}
for name, make_model in bn_dropout_configs.items():
    seed_everything(42)
    model = make_model().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    bn_dropout_histories[name] = fit(
        model, short_train_loader, short_val_loader,
        criterion, optimizer, device, epochs=3
    )

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, history in bn_dropout_histories.items():
    axes[0].plot(history["val_loss"], label=name)
    axes[1].plot(history["val_acc"], label=name)
axes[0].set_title("Validation Loss: BatchNorm/Dropout Off vs On")
axes[0].legend()
axes[1].set_title("Validation Accuracy: BatchNorm/Dropout Off vs On")
axes[1].legend()
plt.show()

for name, history in bn_dropout_histories.items():
    print(f"{name}: 최종 검증 정확도={history['val_acc'][-1]:.2f}%, 최종 검증 손실={history['val_loss'][-1]:.4f}")


## BatchNorm, Dropout, Early Stopping
지금까지는 `use_batchnorm=False, dropout=0.0`인 `ImprovedCNN`만 사용했습니다. 이제 `use_batchnorm=True, dropout=0.3`으로 배치 정규화와 드롭아웃을 함께 켠 모델을 학습하면서, 검증 손실이 더 이상 좋아지지 않을 때 학습을 멈추고 **가장 좋았던 가중치**로 되돌리는 `EarlyStopping`을 사용합니다. `EarlyStopping`은 가중치를 파일로 저장하지 않고, `state_dict()`를 CPU 텐서로 복제해 메모리에만 보관합니다.

### 관찰 질문
- `patience=3`은 검증 손실이 몇 번 연속으로 개선되지 않아야 학습이 멈춘다는 뜻일까요?
- 왜 최적 가중치를 파일에 저장했다가 다시 읽는 대신, 메모리에 있는 `state_dict()` 복제본을 그대로 사용하는 방식을 택했을까요?


In [ ]:
class EarlyStopping:
    def __init__(self, patience=3, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.bad_epochs = 0
        self.best_state = None

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.bad_epochs = 0
            self.best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            return False
        self.bad_epochs += 1
        return self.bad_epochs >= self.patience

    def restore(self, model):
        model.load_state_dict(self.best_state)


## 최종 모델 학습(전체 데이터)
최종 모델은 `ImprovedCNN(use_batchnorm=True, dropout=0.3)`을 `Adam(lr=1e-3)`으로 최대 12에포크, `EarlyStopping(patience=3)`과 함께 **전체 훈련 분할**(`train_loader`, 45,000개, 데이터 증강 포함)에서 학습합니다. 짧은 비교 실험과 달리 여기서는 지금까지 관찰한 설정(Adam, 적절한 학습률, 배치 정규화, 드롭아웃)을 실제 규모의 데이터에 적용합니다.


In [ ]:
seed_everything(42)
final_model = ImprovedCNN(use_batchnorm=True, dropout=0.3).to(device)
final_optimizer = torch.optim.Adam(final_model.parameters(), lr=1e-3)
early_stopping = EarlyStopping(patience=3, min_delta=0.0)
max_epochs = 12
final_history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(1, max_epochs + 1):
    train_loss, train_accuracy = train_one_epoch(final_model, train_loader, criterion, final_optimizer, device)
    val_loss, val_accuracy = evaluate(final_model, val_loader, criterion, device)
    for key, value in [("train_loss", train_loss), ("train_acc", train_accuracy), ("val_loss", val_loss), ("val_acc", val_accuracy)]:
        final_history[key].append(value)
    print(f"Epoch {epoch}/{max_epochs}: train={train_accuracy:.2f}%, val={val_accuracy:.2f}%, val_loss={val_loss:.4f}")
    stop_now = early_stopping.step(val_loss, final_model)
    if stop_now:
        print(f"검증 손실이 {early_stopping.patience}번 연속 개선되지 않아 {epoch} 에포크에서 조기 종료합니다.")
        break

early_stopping.restore(final_model)
plot_history(final_history, "Final ImprovedCNN (BatchNorm+Dropout)")


## 최종 평가: 테스트 정확도와 혼동 행렬
Early Stopping이 메모리에서 복원한 최적 가중치로 테스트 정확도를 계산하고, `sklearn.metrics.confusion_matrix`로 클래스 간 혼동 양상을 확인합니다. 대각선이 아닌 칸의 값이 크다면 두 클래스가 자주 헷갈린다는 뜻입니다. 클래스별 정확도 막대그래프로 어떤 클래스가 특히 어려운지도 살펴봅니다.


In [ ]:
final_test_loss, final_test_accuracy = evaluate(final_model, test_loader, criterion, device)
print(f"최종 모델 테스트 손실: {final_test_loss:.4f}, 테스트 정확도: {final_test_accuracy:.2f}%")

final_model.eval()
all_predictions, all_labels = [], []
with torch.no_grad():
    for batch_images, batch_labels in test_loader:
        logits = final_model(batch_images.to(device))
        all_predictions.append(logits.argmax(dim=1).cpu())
        all_labels.append(batch_labels)
all_predictions = torch.cat(all_predictions).numpy()
all_labels = torch.cat(all_labels).numpy()

confusion = confusion_matrix(all_labels, all_predictions)
fig, axis = plt.subplots(figsize=(7, 6))
image_handle = axis.imshow(confusion, cmap="Blues")
axis.set_xticks(range(10))
axis.set_xticklabels(class_names, rotation=45, ha="right")
axis.set_yticks(range(10))
axis.set_yticklabels(class_names)
axis.set_xlabel("Predicted class")
axis.set_ylabel("Actual class")
axis.set_title("Confusion Matrix")
plt.colorbar(image_handle, ax=axis)
plt.tight_layout()
plt.show()

per_class_accuracy = confusion.diagonal() / confusion.sum(axis=1)
fig, axis = plt.subplots(figsize=(9, 4))
axis.bar(class_names, per_class_accuracy)
axis.set_ylabel("Accuracy")
axis.set_title("Test Accuracy by Class")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
for name, accuracy in zip(class_names, per_class_accuracy):
    print(f"{name}: {accuracy:.2%}")


## 미니 챌린지와 정리

### 학생 활동
1. 학습률 `3e-4`와 `1e-3` 중 하나를 선택하세요.
2. Dropout `0.2`와 `0.4` 중 하나를 선택하세요.
3. 오늘 관찰한 학습률 비교, BatchNorm·Dropout 효과를 근거로 왜 그 조합을 선택했는지 2~3문장으로 설명하세요.
4. 선택한 조합 **하나만** 아래 코드의 `chosen_lr`, `chosen_dropout`에 넣어 학습하고 오늘의 최종 모델과 비교하세요. 네 가지 조합을 모두 자동으로 돌리지 않습니다. 그렇게 하면 실습 시간 안에 결과를 확인하기 어렵습니다.

오늘 다룬 학습률, 옵티마이저, 배치 정규화, 드롭아웃, Early Stopping은 모두 "하이퍼파라미터 하나를 바꾸고 나머지는 고정해 관찰"하는 절차였습니다. 새로운 데이터셋을 만나도 같은 절차를 그대로 적용해 볼 수 있습니다.


In [ ]:
chosen_lr = 3e-4  # 3e-4 또는 1e-3 중 선택
chosen_dropout = 0.2  # 0.2 또는 0.4 중 선택

seed_everything(42)
challenge_model = ImprovedCNN(use_batchnorm=True, dropout=chosen_dropout).to(device)
challenge_optimizer = torch.optim.Adam(challenge_model.parameters(), lr=chosen_lr)
challenge_early_stopping = EarlyStopping(patience=3, min_delta=0.0)
challenge_history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(1, max_epochs + 1):
    train_loss, train_accuracy = train_one_epoch(challenge_model, train_loader, criterion, challenge_optimizer, device)
    val_loss, val_accuracy = evaluate(challenge_model, val_loader, criterion, device)
    for key, value in [("train_loss", train_loss), ("train_acc", train_accuracy), ("val_loss", val_loss), ("val_acc", val_accuracy)]:
        challenge_history[key].append(value)
    print(f"Epoch {epoch}/{max_epochs}: train={train_accuracy:.2f}%, val={val_accuracy:.2f}%")
    if challenge_early_stopping.step(val_loss, challenge_model):
        print(f"{epoch} 에포크에서 조기 종료합니다.")
        break

challenge_early_stopping.restore(challenge_model)
challenge_test_loss, challenge_test_accuracy = evaluate(challenge_model, test_loader, criterion, device)
print(f"챌린지 모델(lr={chosen_lr}, dropout={chosen_dropout}) 테스트 정확도: {challenge_test_accuracy:.2f}%")
print(f"오늘의 최종 모델 테스트 정확도: {final_test_accuracy:.2f}%")
